In [ ]:
!pip install -q rank-bm25 rapidfuzz rdflib sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 19.3 MB/s eta 0:00:00


In [ ]:
import os, re, json, pickle, torch, numpy as np
from collections import defaultdict
from rdflib import Graph, RDFS
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import fuzz

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

from google.colab import files
uploaded = files.upload()

Saving emb_class.pt to emb_class.pt
Saving emb_prop.pt to emb_prop.pt
Saving metadata_class.json to metadata_class.json
Saving metadata_prop.json to metadata_prop.json
Saving dbpedia.owl to dbpedia.owl


## Instance Retriever

In [ ]:

DATA_DIR      = "/content/drive/MyDrive/dbpedia_clean"
SCHEMA_CACHE  = "."
MODEL_NAME    = "all-MiniLM-L6-v2"
ONTOLOGY_PATH = "dbpedia.owl"
TOP_K_SCHEMA  = 10
BOOST         = 0.1

with open(f"{DATA_DIR}/corpus.pkl",       "rb") as f: corpus      = pickle.load(f)
with open(f"{DATA_DIR}/uri_list.json")        as f: uri_list     = json.load(f)
with open(f"{DATA_DIR}/label_to_uri.json")    as f: label_to_uri = json.load(f)
with open(f"{DATA_DIR}/bm25.pkl",         "rb") as f: bm25        = pickle.load(f)
print(" Instance retriever files loaded from Drive")

def _tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def retrieve_instance(query, top_k=1):
    query_clean = query.lower().strip()
    if query_clean in label_to_uri:
        return label_to_uri[query_clean]
    tokens  = _tokenize(query_clean)
    scores  = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:1000]
    results = []
    for i in top_idx:
        base_score   = float(scores[i])
        label        = " ".join(corpus[i])
        token_sim    = fuzz.token_sort_ratio(query_clean, label)
        string_sim   = fuzz.ratio(query_clean, label)
        label_tokens = corpus[i]
        matched      = sum(1 for qt in tokens if any(fuzz.ratio(qt, lt) >= 80 for lt in label_tokens))
        coverage     = matched / len(tokens) if tokens else 0
        len_sim      = 1 - abs(len(label) - len(query_clean)) / max(len(label), len(query_clean), 1)
        final_score  = (
            base_score  * 1.0
            + token_sim  * 0.4
            + string_sim * 0.2
            + coverage   * 20.0
            + len_sim    * 5.0
        )
        results.append((i, final_score))
    results.sort(key=lambda x: x[1], reverse=True)
    top = results[:top_k]
    if not top: return None
    return uri_list[top[0][0]]


 Instance retriever files loaded from Drive


## Schema Retriever

In [ ]:

def _parse_plan_string(plan_str):
    steps = []
    for i, step_str in enumerate(plan_str.split("||"), start=1):
        step_str = step_str.strip()
        if not step_str: continue
        step_str = re.sub(r"^step\d+:\s*", "", step_str)
        step = {"step": i}
        for part in step_str.split("|"):
            part = part.strip()
            if ":" in part:
                k, v = part.split(":", 1)
                step[k.strip()] = v.strip()
        steps.append(step)
    return steps

def _normalise_plan(plan):
    if isinstance(plan, str):  return _parse_plan_string(plan)
    if isinstance(plan, dict) and "steps" in plan: return plan["steps"]
    if isinstance(plan, list): return plan
    raise ValueError(f"Unsupported plan type: {type(plan)}")

def _extract_subj_obj(step):
    action    = step.get("action", "")
    join_type = step.get("join_type", "")
    if action == "find_object":
        if join_type == "intersect": return step.get("subject_variable"), step.get("filter_variable")
        return step.get("subject_variable"), step.get("output_variable")
    elif action == "find_subjects": return step.get("output_variable"), step.get("object_variable")
    elif action in ("find_statement", "left_join", "optional_find"): return step.get("subject_variable"), step.get("output_variable")
    elif action == "find_qualifier": return step.get("subject_variable"), step.get("output_variable")
    elif action == "filter_statement": return step.get("filter_variable"), step.get("value_variable") or step.get("value")
    elif action == "verify_fact": return step.get("subject_variable"), step.get("object_variable")
    elif action == "property_path": return step.get("subject_variable"), step.get("object_variable")
    elif action in ("exists", "exclude"): return step.get("filter_variable"), None
    return None, None

def extract_schema_with_variables(plan):
    steps = _normalise_plan(plan)
    properties, classes = [], []
    for step in steps:
        action    = step.get("action", "")
        prop_name = step.get("property") or step.get("qualifier_property") or step.get("property_path")
        if prop_name:
            subj, obj = _extract_subj_obj(step)
            prop_data = {"property": prop_name, "subject": subj, "object": obj}
            if prop_data not in properties: properties.append(prop_data)
        if action in ("find_by_type", "filter_type") and "type" in step:
            class_data = {"class": step["type"], "variable": step.get("output_variable") or step.get("filter_variable")}
            if class_data not in classes: classes.append(class_data)
        if action == "verify_fact" and "object_type" in step:
            class_data = {"class": step["object_type"], "variable": step.get("object_variable") or step.get("subject_variable")}
            if class_data not in classes: classes.append(class_data)
    return properties, classes

class TextToSPARQLRetriever:
    _FULL_TO_SHORT = {
        "http://dbpedia.org/ontology/":                "dbo:",
        "http://dbpedia.org/property/":                "dbp:",
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#": "rdf:",
        "http://www.w3.org/2000/01/rdf-schema#":       "rdfs:",
        "http://www.w3.org/2001/XMLSchema#":           "xsd:",
    }
    _SHORT_TO_FULL = {v: k for k, v in _FULL_TO_SHORT.items()}

    def __init__(self, cache_dir=SCHEMA_CACHE, model_name=MODEL_NAME, ontology_path=ONTOLOGY_PATH):
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        print("Loading schema cache...")
        self.prop_embeddings  = torch.load(os.path.join(cache_dir, "emb_prop.pt"),   map_location="cpu")
        self.class_embeddings = torch.load(os.path.join(cache_dir, "emb_class.pt"),  map_location="cpu")
        with open(os.path.join(cache_dir, "metadata_prop.json"),  encoding="utf-8") as f: self.prop_metadata  = json.load(f)
        with open(os.path.join(cache_dir, "metadata_class.json"), encoding="utf-8") as f: self.class_metadata = json.load(f)
        print(f"  {len(self.prop_metadata)} properties | {len(self.class_metadata)} classes loaded")
        print(f"Loading ontology from {ontology_path}...")
        g = Graph()
        g.parse(ontology_path)
        self.subclass_map = defaultdict(set)
        for child, parent in g.subject_objects(RDFS.subClassOf):
            self.subclass_map[str(child)].add(str(parent))


    def _normalize(self, uri):
        if not uri: return ""
        for full, short in self._FULL_TO_SHORT.items():
            if uri.startswith(full): return uri.replace(full, short, 1)
        return uri

    def _expand_uri(self, short):
        if not short: return short
        for prefix, full in self._SHORT_TO_FULL.items():
            if short.startswith(prefix): return short.replace(prefix, full, 1)
        return short

    def _is_subclass(self, child, parent, visited=None):
        if child == parent: return True
        if visited is None: visited = set()
        if child in visited: return False
        visited.add(child)
        for p in self.subclass_map.get(child, []):
            if self._is_subclass(p, parent, visited): return True
        return False

    def _compatible(self, range_uri, domain_uri):
        if not range_uri or not domain_uri: return False
        r, d = self._expand_uri(range_uri), self._expand_uri(domain_uri)
        return r == d or self._is_subclass(r, d) or self._is_subclass(d, r)

    def _get_top_k_props(self, label, k=10):
        query_emb = self.model.encode(label, convert_to_tensor=True)
        scores    = util.cos_sim(query_emb, self.prop_embeddings)[0]
        top       = torch.topk(scores, k=k)
        results   = []
        for score, idx in zip(top.values, top.indices):
            entry = self.prop_metadata[idx.item()]
            results.append({"uri": self._normalize(entry.get("uri","")), "domain": self._normalize(entry.get("domain","")), "range": self._normalize(entry.get("range","")), "score": score.item()})
        return results

    def _get_top_k_classes(self, label, k=10):
        query_emb = self.model.encode(label, convert_to_tensor=True)
        scores    = util.cos_sim(query_emb, self.class_embeddings)[0]
        top       = torch.topk(scores, k=k)
        results   = []
        for score, idx in zip(top.values, top.indices):
            entry = self.class_metadata[idx.item()]
            results.append({"uri": self._normalize(entry.get("uri","")), "score": score.item()})
        return results

    def _rerank_properties(self, plan_props, results, boost=0.1):
        results = {k: [dict(c) for c in v] for k, v in results.items()}
        for p1 in plan_props:
            for p2 in plan_props:
                link_var = p1["object"]
                if not link_var or link_var != p2["subject"]: continue
                boost_p1, boost_p2 = set(), set()
                for c1 in results[p1["property"]]:
                    for c2 in results[p2["property"]]:
                        if self._compatible(c1["range"], c2["domain"]):
                            boost_p1.add(c1["uri"]); boost_p2.add(c2["uri"])
                for c in results[p1["property"]]:
                    if c["uri"] in boost_p1: c["score"] += boost
                for c in results[p2["property"]]:
                    if c["uri"] in boost_p2: c["score"] += boost
        for key in results: results[key].sort(key=lambda x: x["score"], reverse=True)
        return results

    def process_plan(self, plan, top_k=10, boost=0.1):
        plan_props, plan_classes = extract_schema_with_variables(plan)
        prop_raw  = {p["property"]: self._get_top_k_props(p["property"], k=top_k) for p in plan_props}
        class_raw = {c["class"]:    self._get_top_k_classes(c["class"],  k=top_k) for c in plan_classes}
        prop_results = self._rerank_properties(plan_props, prop_raw, boost=boost)
        return {"properties": prop_results, "classes": class_raw}

retriever = TextToSPARQLRetriever(
    cache_dir=SCHEMA_CACHE,
    model_name=MODEL_NAME,
    ontology_path=ONTOLOGY_PATH,
)

Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading schema cache...
  3029 properties | 790 classes loaded
Loading ontology from dbpedia.owl...


In [ ]:

def shorten_uri(text):
    text = re.sub(r"http://dbpedia\.org/resource/([A-Za-z0-9_%(),\.\-]+)",  r"dbr:\1", text)
    text = re.sub(r"http://dbpedia\.org/ontology/([A-Za-z0-9_%(),\.\-]+)",  r"dbo:\1", text)
    text = re.sub(r"http://dbpedia\.org/property/([A-Za-z0-9_%(),\.\-]+)",  r"dbp:\1", text)
    return text

def extract_entities_from_plan(plan_str):
    entities = []
    for step_str in plan_str.split("||"):
        if "find_entity" in step_str:
            sf_m  = re.search(r'surface_form:\s*([^|]+)', step_str)
            var_m = re.search(r'output_variable:\s*([^|]+)', step_str)
            if sf_m and var_m:
                entities.append({"surface_form": sf_m.group(1).strip(), "output_variable": var_m.group(1).strip()})
    return entities

def build_transformer_input(question, plan, dataset="DBpedia"):
    # 1. Entity / instance retrieval
    entities = extract_entities_from_plan(plan)
    for e in entities:
        uri = retrieve_instance(e["surface_form"], top_k=1)
        e["retrieved_uri"] = shorten_uri(uri) if uri else e["surface_form"]

    # 2. Schema (property + class) retrieval
    schema_out   = retriever.process_plan(plan, top_k=TOP_K_SCHEMA, boost=BOOST)
    prop_results = schema_out["properties"]
    cls_results  = schema_out["classes"]

    retrieved_props = []
    for label, candidates in prop_results.items():
        top_uri = shorten_uri(candidates[0]["uri"]) if candidates else label
        retrieved_props.append({"label": label, "uri": top_uri})

    retrieved_classes = []
    for label, candidates in cls_results.items():
        top_uri = shorten_uri(candidates[0]["uri"]) if candidates else label
        retrieved_classes.append({"label": label, "uri": top_uri})

    # 3. Build schema string
    schema_lines = []
    for e in entities:
        schema_lines.append(f"  {e['surface_form']} -> {e['retrieved_uri']}")
    for p in retrieved_props:
        schema_lines.append(f"  {p['label']} -> {p['uri']}")
    for c in retrieved_classes:
        schema_lines.append(f"  {c['label']} -> {c['uri']}")
    schema_str = "\n".join(schema_lines)


    final_input = (
        f"generate sparql [{dataset}]: {question}\n"
        f"plan: {plan}\n"
        f"schema:\n{schema_str}"
    )
    return final_input


In [ ]:


question = "Which governing body of the Oahu Railway and Land Company is also the military branch of the Jimmy Quillen?"

plan = "step1: action: find_entity | surface_form: Jimmy Quillen | output_variable: ?jimmy_quillen | semantic_type: ENTITY || step2: action: find_entity | surface_form: Oahu Railway and Land Company | output_variable: ?oahu_railway_and_lan | semantic_type: ENTITY || step3: action: find_object | property: militaryBranch | subject_variable: ?jimmy_quillen | output_variable: ?uri | semantic_type: PROPERTY || step4: action: find_object | property: governingBody | subject_variable: ?oahu_railway_and_lan | semantic_type: PROPERTY | join_type: intersect | filter_variable: ?uri || step5: action: distinct | semantic_type: MODIFIER"

dataset = "DBpedia"


final_input = build_transformer_input(question, plan, dataset)



print(final_input)


generate sparql [DBpedia]: Which governing body of the Oahu Railway and Land Company is also the military branch of the Jimmy Quillen?
plan: step1: action: find_entity | surface_form: Jimmy Quillen | output_variable: ?jimmy_quillen | semantic_type: ENTITY || step2: action: find_entity | surface_form: Oahu Railway and Land Company | output_variable: ?oahu_railway_and_lan | semantic_type: ENTITY || step3: action: find_object | property: militaryBranch | subject_variable: ?jimmy_quillen | output_variable: ?uri | semantic_type: PROPERTY || step4: action: find_object | property: governingBody | subject_variable: ?oahu_railway_and_lan | semantic_type: PROPERTY | join_type: intersect | filter_variable: ?uri || step5: action: distinct | semantic_type: MODIFIER
schema:
  Jimmy Quillen -> dbr:Jimmy_Quillen
  Oahu Railway and Land Company -> dbr:Oahu_Railway_and_Land_Company
  militaryBranch -> dbo:militaryBranch
  governingBody -> dbo:governingBody
